# Notebook 5 — Finalisasi Model dan Ekspor ke Laptop

Notebook ini melatih dua kandidat klasifikasi yang sesuai proposal (KNN dan Random Forest), K-Means untuk clustering, Isolation Forest untuk deteksi kandidat anomali, menyimpan artefak serta fungsi inferensi, lalu mengunduh satu ZIP ke laptop melalui Google Colab.

**Batasan:** ini bukan model peramalan. Dataset fishpond belum membuktikan performa pada sungai nyata; label yang terkait erat dengan kekeruhan dapat membuat skor klasifikasi terlalu optimistis. Anomali bukan bukti pasti pencemaran.

## 1. Unggah dataset eksperimen
Unggah `dataset_eksperimen_sensor.csv` dari Notebook 3. Kolom wajib: `temp_C`, `pH`, `turbidity_NTU`, `class`.

In [ ]:
import io, json, shutil
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from google.colab import files

uploaded = files.upload()
if not uploaded: raise ValueError('Dataset belum diunggah')
name = next((n for n in uploaded if n.lower().endswith('.csv')), None)
if name is None: raise ValueError('Unggah file CSV')
df = pd.read_csv(io.BytesIO(uploaded[name]))
df.columns = [str(c).strip() for c in df.columns]
print('File:', name, '| Ukuran:', df.shape)
display(df.head())

## 2. Siapkan fitur sensor
Hanya tiga fitur sesuai sensor proyek. Baris dengan fitur/target kosong atau tak hingga dikeluarkan. Outlier tidak otomatis dihapus karena bisa relevan untuk deteksi anomali.

In [ ]:
FEATURES = ['temp_C', 'pH', 'turbidity_NTU']; TARGET = 'class'
missing = [c for c in FEATURES + [TARGET] if c not in df.columns]
if missing: raise ValueError(f'Kolom wajib tidak ditemukan: {missing}')
work = df[FEATURES + [TARGET]].copy()
for c in FEATURES: work[c] = pd.to_numeric(work[c], errors='coerce')
work[TARGET] = work[TARGET].astype('string').str.strip()
work = work.replace([np.inf, -np.inf], np.nan).dropna(subset=FEATURES + [TARGET]).reset_index(drop=True)
X = work[FEATURES]; y = work[TARGET].astype(str)
print('Baris siap pakai:', len(work)); display(work[TARGET].value_counts().to_frame('jumlah'))

## 3. Latih dan evaluasi KNN serta Random Forest
Stratified train/test split digunakan sebagai evaluasi awal. Skor tidak membuktikan generalisasi ke sungai nyata.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, ConfusionMatrixDisplay

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
models = {'KNN': Pipeline([('scaler', StandardScaler()), ('model', KNeighborsClassifier(n_neighbors=5))]), 'Random Forest': RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced')}
rows=[]; reports={}
for model_name, model in models.items():
    model.fit(X_train, y_train); pred=model.predict(X_test)
    rows.append({'model':model_name,'accuracy':accuracy_score(y_test,pred),'precision_macro':precision_score(y_test,pred,average='macro',zero_division=0),'recall_macro':recall_score(y_test,pred,average='macro',zero_division=0),'f1_macro':f1_score(y_test,pred,average='macro',zero_division=0)})
    reports[model_name]=classification_report(y_test,pred,output_dict=True,zero_division=0)
comparison=pd.DataFrame(rows).sort_values('f1_macro',ascending=False).reset_index(drop=True)
display(comparison)
ax=comparison.set_index('model')[['accuracy','precision_macro','recall_macro','f1_macro']].plot(kind='bar',figsize=(8,4),ylim=(0,1.05),title='Perbandingan KNN dan Random Forest')
ax.grid(axis='y',alpha=.25); plt.tight_layout(); plt.show()
for model_name, model in models.items():
    ConfusionMatrixDisplay.from_predictions(y_test,model.predict(X_test),labels=sorted(y.unique()),xticks_rotation=45)
    plt.title('Confusion Matrix — '+model_name); plt.tight_layout(); plt.show()

## 4. Tetapkan baseline klasifikasi
Model dengan macro-F1 tertinggi dipilih sebagai baseline. Setelah metrik test dicatat, model dilatih ulang menggunakan seluruh data eksperimen untuk prototipe.

In [ ]:
best_name = str(comparison.iloc[0]['model'])
best_model = models[best_name]
best_model.fit(X, y)
print('Model baseline terpilih:', best_name)

## 5. Latih K-Means dan Isolation Forest
K-Means mengelompokkan pola sensor; cluster tidak otomatis berarti bersih/tercemar. Isolation Forest menandai kandidat anomali, bukan memastikan pencemaran.

In [ ]:
scaler_clustering = StandardScaler(); X_cluster = scaler_clustering.fit_transform(X)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_ids = kmeans.fit_predict(X_cluster)
hasil_clustering = work.copy(); hasil_clustering['cluster_id'] = cluster_ids
display(pd.crosstab(hasil_clustering['cluster_id'], hasil_clustering[TARGET]))
scaler_anomaly = StandardScaler(); X_anomaly = scaler_anomaly.fit_transform(X)
isolation_forest = IsolationForest(n_estimators=200, contamination='auto', random_state=42)
anom_pred = isolation_forest.fit_predict(X_anomaly)
hasil_anomali = work.copy(); hasil_anomali['anomaly_label'] = np.where(anom_pred == -1, 'Anomali', 'Normal')
hasil_anomali['anomaly_score'] = isolation_forest.decision_function(X_anomaly)
display(hasil_anomali['anomaly_label'].value_counts().to_frame('jumlah'))

## 6. Buat dan uji fungsi inferensi

In [ ]:
def analyze_reading(reading: dict) -> dict:
    if not isinstance(reading, dict): raise TypeError('Input harus dictionary')
    missing = [c for c in FEATURES if c not in reading]
    if missing: raise ValueError(f'Fitur wajib tidak ada: {missing}')
    vals = {}
    for c in FEATURES:
        try: vals[c] = float(reading[c])
        except (TypeError, ValueError): raise ValueError(f'Nilai {c} harus numerik')
        if not np.isfinite(vals[c]): raise ValueError(f'Nilai {c} harus finite')
    one = pd.DataFrame([vals], columns=FEATURES)
    z = scaler_anomaly.transform(one)
    return {'classification_model': best_name, 'class_label': str(best_model.predict(one)[0]), 'cluster_id': int(kmeans.predict(scaler_clustering.transform(one))[0]), 'anomaly_detected': bool(isolation_forest.predict(z)[0] == -1), 'anomaly_score': float(isolation_forest.decision_function(z)[0]), 'interpretation': 'Indikasi untuk pemeriksaan; bukan bukti pasti pencemaran.'}
sample = {c: float(X.iloc[0][c]) for c in FEATURES}
print(analyze_reading(sample))
assert 'class_label' in analyze_reading(sample)
for bad in [{'temp_C': 28, 'pH': 7}, {'temp_C': 'abc', 'pH': 7, 'turbidity_NTU': 20}, {'temp_C': np.nan, 'pH': 7, 'turbidity_NTU': 20}]:
    try: analyze_reading(bad)
    except (ValueError, TypeError): print('Uji input tidak valid: LULUS')

## 7. Simpan model, scaler, hasil, dan modul inferensi

In [ ]:
import joblib
OUT = Path('/content/output_model_integrasi_smart_aquatic'); OUT.mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, OUT/'model_klasifikasi_baseline.joblib')
joblib.dump(models['KNN'], OUT/'model_knn.joblib')
joblib.dump(models['Random Forest'], OUT/'model_random_forest.joblib')
joblib.dump(kmeans, OUT/'model_kmeans.joblib'); joblib.dump(scaler_clustering, OUT/'scaler_clustering.joblib')
joblib.dump(isolation_forest, OUT/'model_isolation_forest.joblib'); joblib.dump(scaler_anomaly, OUT/'scaler_isolation_forest.joblib')
comparison.to_csv(OUT/'perbandingan_model_klasifikasi.csv', index=False)
hasil_clustering.to_csv(OUT/'hasil_clustering.csv', index=False); hasil_anomali.to_csv(OUT/'hasil_deteksi_anomali.csv', index=False)
with open(OUT/'classification_reports.json','w',encoding='utf-8') as f: json.dump(reports,f,indent=2,ensure_ascii=False)
summary = {'project':'Smart Aquatic','purpose':'Deteksi/klasifikasi kondisi saat ini, bukan forecasting','features':FEATURES,'target':TARGET,'selected_classifier':best_name,'rows_used':int(len(work)),'test_metrics':comparison.to_dict(orient='records'),'limitations':['Dataset fishpond belum membuktikan performa pada sungai nyata.','Label berpotensi berkaitan langsung dengan kekeruhan.','Anomali bukan bukti pasti pencemaran.']}
with open(OUT/'ringkasan_eksperimen.json','w',encoding='utf-8') as f: json.dump(summary,f,indent=2,ensure_ascii=False)
source = '''from pathlib import Path
import joblib, numpy as np, pandas as pd
BASE = Path(__file__).resolve().parent
FEATURES = ["temp_C", "pH", "turbidity_NTU"]
clf = joblib.load(BASE / "model_klasifikasi_baseline.joblib")
km = joblib.load(BASE / "model_kmeans.joblib")
sc = joblib.load(BASE / "scaler_clustering.joblib")
iso = joblib.load(BASE / "model_isolation_forest.joblib")
sa = joblib.load(BASE / "scaler_isolation_forest.joblib")
def analyze_reading(reading):
    if not isinstance(reading, dict): raise TypeError("Input harus dictionary")
    missing = [c for c in FEATURES if c not in reading]
    if missing: raise ValueError(f"Fitur wajib tidak ada: {missing}")
    vals = {c: float(reading[c]) for c in FEATURES}
    if not all(np.isfinite(v) for v in vals.values()): raise ValueError("Nilai harus finite")
    one = pd.DataFrame([vals], columns=FEATURES); z = sa.transform(one)
    return {"class_label": str(clf.predict(one)[0]), "cluster_id": int(km.predict(sc.transform(one))[0]), "anomaly_detected": bool(iso.predict(z)[0] == -1), "anomaly_score": float(iso.decision_function(z)[0]), "interpretation": "Indikasi; bukan bukti pasti pencemaran"}
'''
(OUT/'inference.py').write_text(source,encoding='utf-8')
(OUT/'README.txt').write_text('Smart Aquatic model artifacts. Features: temp_C, pH, turbidity_NTU. Output is indicative only, not proof of pollution. No future forecasting. Install compatible pandas, numpy, scikit-learn, joblib.',encoding='utf-8')
print('Output:', sorted(p.name for p in OUT.iterdir()))

## 8. Buat ZIP dan unduh ke laptop
Jalankan sel terakhir. Browser akan mengunduh ZIP ke laptop, biasanya folder Downloads.

In [ ]:
archive = shutil.make_archive('/content/Model_Integrasi_SmartAquatic', 'zip', OUT)
print('ZIP dibuat:', archive)
files.download(archive)

## Checklist
- [ ] Semua sel berjalan tanpa error.
- [ ] ZIP terunduh ke laptop dan diekstrak.
- [ ] Folder berisi model, scaler, hasil CSV, ringkasan JSON, README, dan `inference.py`.
- [ ] Backend memakai nama fitur yang sama.
- [ ] Output diperlakukan sebagai indikasi, bukan bukti pasti pencemaran.